# Smart Health Assistant - Multi-Agent System
Uses AutoGen sequential conversation pattern with Groq API

In [1]:
import os
from google.colab import userdata

# Get Groq API key from Colab secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

print("Groq API Key loaded successfully!")

Groq API Key loaded successfully!


In [10]:
!pip install autogen -q
!pip install openai diskcache termcolor tiktoken -q

import importlib
import sys

# Remove cached broken module
for mod in list(sys.modules.keys()):
    if 'autogen' in mod:
        del sys.modules[mod]

import autogen
from autogen import AssistantAgent, UserProxyAgent, register_function

# LLM Configuration using Groq
llm_config = {
    "config_list": [
        {
            "model": "llama-3.3-70b-versatile",
            "api_key": GROQ_API_KEY,
            "base_url": "https://api.groq.com/openai/v1",
            "api_type": "openai",
        }
    ],
    "temperature": 0.3,
}

print("LLM configuration set!")

LLM configuration set!


In [11]:
# ─────────────────────────────────────────────
# BMI Tool Definition
# ─────────────────────────────────────────────
def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    """
    Calculate BMI from weight (kg) and height (cm).

    Args:
        weight_kg: Weight in kilograms
        height_cm: Height in centimeters

    Returns:
        Dictionary with BMI value and category
    """
    # Convert height from cm to m
    height_m = height_cm / 100

    # Calculate BMI
    bmi = weight_kg / (height_m ** 2)
    bmi = round(bmi, 2)

    # Determine BMI category
    if bmi < 18.5:
        category = "Underweight"
    elif 18.5 <= bmi < 25.0:
        category = "Normal weight"
    elif 25.0 <= bmi < 30.0:
        category = "Overweight"
    else:
        category = "Obese"

    result = {
        "weight_kg": weight_kg,
        "height_cm": height_cm,
        "height_m": round(height_m, 4),
        "bmi": bmi,
        "category": category
    }

    print(f"\n📊 BMI Calculation Results:")
    print(f"   Weight: {weight_kg} kg")
    print(f"   Height: {height_cm} cm ({height_m} m)")
    print(f"   BMI: {bmi}")
    print(f"   Category: {category}\n")

    return result

print("BMI tool defined!")

BMI tool defined!


In [12]:
# ─────────────────────────────────────────────
# Agent Definitions
# ─────────────────────────────────────────────

# 1. User Proxy Agent - Collects user data and initiates conversation
user_proxy = UserProxyAgent(
    name="UserProxyAgent",
    system_message="""You are a User Proxy Agent responsible for collecting health information from the user.
You have collected the following user data and will share it with other agents:
- Weight (kg)
- Height (cm)
- Age
- Gender
- Dietary Preference (Veg/Non-Veg/Vegan)

You will pass this information to the BMI Agent to begin the health assessment process.
Do not ask for more information - use what has been provided.""",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=1,
    code_execution_config=False,
    is_termination_msg=lambda x: "HEALTH_PLAN_COMPLETE" in x.get("content", ""),
)

# 2. BMI Agent - Calculates BMI using the tool and provides health recommendations
bmi_agent = AssistantAgent(
    name="BMIAgent",
    system_message="""You are the BMI Agent. Your responsibilities:
1. Use the calculate_bmi tool with the user's weight and height to compute their BMI.
2. Analyze the BMI result and categorize it (Underweight/Normal/Overweight/Obese).
3. Provide clear health recommendations based on the BMI category.
4. Pass your BMI insights along with the user's dietary preference to the Diet Planner Agent.

BMI Categories:
- Underweight: BMI < 18.5
- Normal weight: 18.5 ≤ BMI < 25
- Overweight: 25 ≤ BMI < 30
- Obese: BMI ≥ 30

Always call the calculate_bmi function first before providing recommendations.
After your analysis, explicitly say 'Passing to Diet Planner Agent' and summarize the key info.""",
    llm_config=llm_config,
)

# 3. Diet Planner Agent - Creates tailored meal plans
diet_planner = AssistantAgent(
    name="DietPlannerAgent",
    system_message="""You are the Diet Planner Agent. Your responsibilities:
1. Receive BMI insights and dietary preference from the BMI Agent.
2. Create a detailed 7-day meal plan tailored to the user's BMI category and dietary preference.
3. Include breakfast, lunch, dinner, and snacks for each day.
4. Ensure the meal plan aligns with health goals (weight loss/gain/maintenance).
5. Pass your meal plan suggestions along with user's age and gender to the Workout Scheduler Agent.

Guidelines:
- Veg: Only vegetarian meals
- Non-Veg: Include meat, fish, eggs
- Vegan: No animal products at all

After the meal plan, explicitly say 'Passing to Workout Scheduler Agent' and summarize key recommendations.""",
    llm_config=llm_config,
    human_input_mode="NEVER" if hasattr(AssistantAgent, 'human_input_mode') else None,
)

# 4. Workout Scheduler Agent - Creates weekly workout plans
workout_scheduler = AssistantAgent(
    name="WorkoutSchedulerAgent",
    system_message="""You are the Workout Scheduler Agent. Your responsibilities:
1. Receive meal plan suggestions and user's age and gender.
2. Create a comprehensive 7-day weekly workout plan.
3. Tailor exercises based on age, gender, and BMI category.
4. Include warm-up, main workout, and cool-down for each day.
5. Specify exercise duration, sets, reps, and rest periods.
6. Include at least one rest day in the week.

Guidelines by BMI:
- Underweight: Focus on strength training and muscle building
- Normal: Balanced cardio and strength training
- Overweight: Focus on cardio and moderate strength training
- Obese: Low-impact cardio, walking, swimming, light strength training

Consider age-appropriate exercises and gender-specific needs.
End your response with 'HEALTH_PLAN_COMPLETE' to signal completion.""",
    llm_config=llm_config,
)

print("All agents defined!")

All agents defined!


In [14]:
# ─────────────────────────────────────────────
# Collect User Data
# ─────────────────────────────────────────────
print("=" * 50)
print("   SMART HEALTH ASSISTANT")
print("=" * 50)
print("\nPlease provide your health information:\n")

weight = float(input("Enter your weight (in kg): "))
height = float(input("Enter your height (in cm): "))
age = int(input("Enter your age: "))
gender = input("Enter your gender (Male/Female/Other): ").strip()
dietary_pref = input("Enter your dietary preference (Veg/Non-Veg/Vegan): ").strip()

print("\n" + "=" * 50)
print("User Data Collected:")
print(f"  Weight: {weight} kg")
print(f"  Height: {height} cm")
print(f"  Age: {age}")
print(f"  Gender: {gender}")
print(f"  Dietary Preference: {dietary_pref}")
print("=" * 50 + "\n")

   SMART HEALTH ASSISTANT

Please provide your health information:

Enter your weight (in kg): 110
Enter your height (in cm): 185
Enter your age: 22
Enter your gender (Male/Female/Other): Male
Enter your dietary preference (Veg/Non-Veg/Vegan): Non-Veg

User Data Collected:
  Weight: 110.0 kg
  Height: 185.0 cm
  Age: 22
  Gender: Male
  Dietary Preference: Non-Veg



In [20]:
# ─────────────────────────────────────────────
# All Agents + Tool Registration + Conversation
# ─────────────────────────────────────────────

from autogen import AssistantAgent, UserProxyAgent

# 1. User Proxy Agent
user_proxy = UserProxyAgent(
    name="UserProxyAgent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
    code_execution_config=False,
    is_termination_msg=lambda x: "HEALTH_PLAN_COMPLETE" in (x.get("content") or ""),
)

# 2. BMI Agent
bmi_agent = AssistantAgent(
    name="BMIAgent",
    system_message="""You are the BMI Agent. Your responsibilities:
1. Use the calculate_bmi tool with the user's weight and height to compute their BMI.
2. Analyze the BMI result and categorize it (Underweight/Normal/Overweight/Obese).
3. Provide clear health recommendations based on the BMI category.

BMI Categories:
- Underweight: BMI < 18.5
- Normal weight: 18.5 <= BMI < 25
- Overweight: 25 <= BMI < 30
- Obese: BMI >= 30

Always call the calculate_bmi function first before providing recommendations.
After your analysis, summarize BMI insights clearly.""",
    llm_config=llm_config,
)

# 3. Diet Planner Agent
diet_planner = AssistantAgent(
    name="DietPlannerAgent",
    system_message="""You are the Diet Planner Agent. Your responsibilities:
1. Receive BMI insights and dietary preference.
2. Create a detailed 7-day meal plan tailored to the user's BMI category and dietary preference.
3. Include breakfast, lunch, dinner, and snacks for each day.
4. Ensure the meal plan aligns with health goals (weight loss/gain/maintenance).

Guidelines:
- Veg: Only vegetarian meals
- Non-Veg: Include meat, fish, eggs
- Vegan: No animal products at all

After the meal plan, summarize key dietary recommendations.""",
    llm_config=llm_config,
)

# 4. Workout Scheduler Agent
workout_scheduler = AssistantAgent(
    name="WorkoutSchedulerAgent",
    system_message="""You are the Workout Scheduler Agent. Your responsibilities:
1. Create a comprehensive 7-day weekly workout plan.
2. Tailor exercises based on age, gender, and BMI category.
3. Include warm-up, main workout, and cool-down for each day.
4. Specify exercise duration, sets, reps, and rest periods.
5. Include at least one rest day.

End your response with exactly: HEALTH_PLAN_COMPLETE""",
    llm_config=llm_config,
)

# ── Register BMI Tool ─────────────────────────────────────────
def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    height_m = height_cm / 100
    bmi = round(weight_kg / (height_m ** 2), 2)
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25.0:
        category = "Normal weight"
    elif bmi < 30.0:
        category = "Overweight"
    else:
        category = "Obese"
    print(f"\n📊 BMI Tool: weight={weight_kg}kg, height={height_cm}cm → BMI={bmi} ({category})\n")
    return {"bmi": bmi, "category": category, "height_m": round(height_m, 4)}

# Register tool: BMI Agent suggests it, User Proxy executes it
bmi_agent.register_for_llm(name="calculate_bmi", description="Calculate BMI from weight in kg and height in cm.")(calculate_bmi)
user_proxy.register_for_execution(name="calculate_bmi")(calculate_bmi)

print("✅ All agents and BMI tool registered!")

# ── Initial Message ───────────────────────────────────────────
initial_message = f"""Hello! I have collected the following user health data:

👤 User Profile:
- Weight: {weight} kg
- Height: {height} cm
- Age: {age} years
- Gender: {gender}
- Dietary Preference: {dietary_pref}

Please calculate the BMI using the calculate_bmi tool, then provide health recommendations."""

print("\nStarting Smart Health Assistant...\n")
print("=" * 60)

# ── Sequential Chats ──────────────────────────────────────────
chat_results = user_proxy.initiate_chats(
    [
        {
            "recipient": bmi_agent,
            "message": initial_message,
            "summary_method": "last_msg",
            "max_turns": 4,
        },
        {
            "recipient": diet_planner,
            "message": f"The user has BMI insights from the previous analysis. Create a tailored 7-day meal plan. User: Age={age}, Gender={gender}, Dietary Preference={dietary_pref}.",
            "summary_method": "last_msg",
            "max_turns": 2,
        },
        {
            "recipient": workout_scheduler,
            "message": f"Based on the diet plan and BMI insights, create a 7-day workout schedule. User: Age={age}, Gender={gender}.",
            "summary_method": "last_msg",
            "max_turns": 2,
        },
    ]
)

print("\n" + "=" * 60)
print("✅ HEALTH PLAN GENERATION COMPLETE")
print("=" * 60)

✅ All agents and BMI tool registered!

Starting Smart Health Assistant...


********************************************************************************
Starting a new chat....

********************************************************************************
UserProxyAgent (to BMIAgent):

Hello! I have collected the following user health data:

👤 User Profile:
- Weight: 110.0 kg
- Height: 185.0 cm
- Age: 22 years
- Gender: Male
- Dietary Preference: Non-Veg

Please calculate the BMI using the calculate_bmi tool, then provide health recommendations.

--------------------------------------------------------------------------------
[autogen.oai.client: 02-19 14:24:53] {738} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.


BMIAgent (to UserProxyAgent):

***** Suggested tool call (g43sdnmfp): calculate_bmi *****
Arguments: 
{"height_cm":185,"weight_kg":110}
**********************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION calculate_bmi...
Call ID: g43sdnmfp
Input arguments: {'height_cm': 185, 'weight_kg': 110}

📊 BMI Tool: weight=110.0kg, height=185.0cm → BMI=32.14 (Obese)


>>>>>>>> EXECUTED FUNCTION calculate_bmi...
Call ID: g43sdnmfp
Input arguments: {'height_cm': 185, 'weight_kg': 110}
Output:
{'bmi': 32.14, 'category': 'Obese', 'height_m': 1.85}
UserProxyAgent (to BMIAgent):

***** Response from calling tool (g43sdnmfp) *****
{'bmi': 32.14, 'category': 'Obese', 'height_m': 1.85}
**************************************************

--------------------------------------------------------------------------------
[autogen.oai.client: 02-19 14:24:54] {738} WARNING - Model llama-3.3-70b-versatile is not 

BMIAgent (to UserProxyAgent):

Based on the calculated BMI of 32.14, the user is categorized as Obese. 

As an Obese individual, the user is at a higher risk of developing chronic diseases such as diabetes, heart disease, and certain types of cancer. To improve overall health, I recommend the following:

1. **Weight Loss**: Aim to lose 0.5-1 kg per week through a combination of diet and exercise. Consult a healthcare professional or a registered dietitian to create a personalized weight loss plan.
2. **Balanced Diet**: Focus on whole, unprocessed foods like vegetables, fruits, whole grains, lean proteins, and healthy fats. Limit sugary drinks, fast food, and processed snacks.
3. **Regular Exercise**: Engage in at least 150 minutes of moderate-intensity aerobic exercise, or 75 minutes of vigorous-intensity aerobic exercise, or a combination of both, per week. Additionally, incorporate strength training exercises to build muscle and boost metabolism.
4. **Monitor Progress**: Regularly tr

BMIAgent (to UserProxyAgent):

Since no new user data was provided, I will wait for further instructions or user health data to calculate the BMI and provide health recommendations. Please provide the necessary user data, including weight and height, to proceed.

--------------------------------------------------------------------------------
UserProxyAgent (to BMIAgent):



--------------------------------------------------------------------------------
[autogen.oai.client: 02-19 14:24:55] {738} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.


BMIAgent (to UserProxyAgent):

***** Suggested tool call (wtywfrxqf): calculate_bmi *****
Arguments: 
{"height_cm":180,"weight_kg":70}
**********************************************************

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (ec07611b-9aee-4ac0-9ba7-767991d0358a): Maximum turns (4) reached

********************************************************************************
Starting a new chat....

********************************************************************************
UserProxyAgent (to DietPlannerAgent):

The user has BMI insights from the previous analysis. Create a tailored 7-day meal plan. User: Age=22, Gender=Male, Dietary Preference=Non-Veg.
Context: 


--------------------------------------------------------------------------------
[autogen.oai.client: 02-19 14:24:57] {738} WARNING - Model llama-3.3-70b-versatile is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_p

DietPlannerAgent (to UserProxyAgent):

Based on the user's age, gender, and dietary preference, I will create a 7-day meal plan. Since the user's BMI insights are not provided, I will assume a general meal plan for a 22-year-old male with a non-vegetarian diet. 

Please note that for a more personalized meal plan, the user's BMI category (underweight, normal weight, overweight, or obese) and specific health goals (weight loss, gain, or maintenance) should be considered.

Here is a sample 7-day meal plan for a 22-year-old male with a non-vegetarian diet:

**Day 1**
- Breakfast: Scrambled eggs with whole wheat toast and avocado (320 calories)
- Lunch: Grilled chicken breast with brown rice and mixed vegetables (500 calories)
- Snack: Greek yogurt with nuts and berries (200 calories)
- Dinner: Baked salmon with quinoa and steamed broccoli (550 calories)

**Day 2**
- Breakfast: Oatmeal with banana, almond milk, and honey (400 calories)
- Lunch: Turkey and avocado wrap with mixed greens (60

DietPlannerAgent (to UserProxyAgent):

Based on the user's age, gender, and dietary preference, I will create a 7-day meal plan. Since the user's BMI insights are not provided, I will assume a general meal plan for a 22-year-old male with a non-vegetarian diet. 

Please note that for a more personalized meal plan, the user's BMI category (underweight, normal weight, overweight, or obese) and specific health goals (weight loss, gain, or maintenance) should be considered.

Here is a sample 7-day meal plan for a 22-year-old male with a non-vegetarian diet:

**Day 1**
- Breakfast: Scrambled eggs with whole wheat toast and avocado (320 calories)
- Lunch: Grilled chicken breast with brown rice and mixed vegetables (500 calories)
- Snack: Greek yogurt with nuts and berries (200 calories)
- Dinner: Baked salmon with quinoa and steamed broccoli (550 calories)

**Day 2**
- Breakfast: Oatmeal with banana, almond milk, and honey (400 calories)
- Lunch: Turkey and avocado wrap with mixed greens (60

WorkoutSchedulerAgent (to UserProxyAgent):

Based on the provided diet plan and assuming a normal weight BMI category for a 22-year-old male, I will create a 7-day workout schedule. Please note that this is a general workout plan and may need to be adjusted based on the user's specific health goals and fitness level.

**Day 1: Chest and Triceps**

* Warm-up: 5-10 minutes of cardio (jogging, jumping jacks, etc.)
* Barbell Bench Press: 3 sets of 8-12 reps, 60-90 seconds rest between sets
* Incline Dumbbell Press: 3 sets of 10-15 reps, 60-90 seconds rest between sets
* Tricep Pushdown: 3 sets of 12-15 reps, 60-90 seconds rest between sets
* Tricep Dips: 3 sets of 12-15 reps, 60-90 seconds rest between sets
* Cool-down: 5-10 minutes of stretching

**Day 2: Back and Biceps**

* Warm-up: 5-10 minutes of cardio
* Pull-ups: 3 sets of 8-12 reps, 60-90 seconds rest between sets
* Barbell Rows: 3 sets of 8-12 reps, 60-90 seconds rest between sets
* Dumbbell Bicep Curls: 3 sets of 12-15 reps, 60-9

In [18]:
# ─────────────────────────────────────────────
# Display Final Summary
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("   ✅ HEALTH PLAN GENERATION COMPLETE")
print("=" * 60)
print(f"""
Your personalized health plan has been created!

📋 Summary of what was generated:
  1. 🔢 BMI Calculation & Health Recommendations (BMI Agent)
  2. 🥗 7-Day Personalized Meal Plan (Diet Planner Agent)
  3. 💪 7-Day Weekly Workout Schedule (Workout Scheduler Agent)

User Profile: {age} year old {gender} | {dietary_pref} diet | {weight}kg @ {height}cm
""")


   ✅ HEALTH PLAN GENERATION COMPLETE

Your personalized health plan has been created!

📋 Summary of what was generated:
  1. 🔢 BMI Calculation & Health Recommendations (BMI Agent)
  2. 🥗 7-Day Personalized Meal Plan (Diet Planner Agent)
  3. 💪 7-Day Weekly Workout Schedule (Workout Scheduler Agent)

User Profile: 22 year old Male | Non-Veg diet | 110.0kg @ 185.0cm

